In [1]:
from langchain.messages import HumanMessage
from typing import TypedDict
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.chat_models import init_chat_model
from agent_lab.model_hub import LLM_FAST

class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get('user_role', 'user') # type: ignore
    base_prompt = 'You are a helpful assistant.'

    if user_role == 'expert':
        return f'{base_prompt} Answer questions briefly, without elaborating on simple concepts.'
    elif user_role == 'beginner':
        return f'{base_prompt} Provide detailed explanations of professional concepts so that even beginners can understand them.'

    return base_prompt

agent = create_agent(
    model=init_chat_model(**LLM_FAST),
    middleware=[user_role_prompt], # type: ignore
    context_schema=Context,
)

In [2]:
expert_response = agent.invoke(
    {'messages': [HumanMessage('Explain machine learning.')]}, context={'user_role': 'expert'}
)
beginner_response = agent.invoke(
    {'messages': [HumanMessage('Explain machine learning.')]}, context={'user_role': 'beginner'}
)

In [3]:
for msg in expert_response['messages']:
    msg.pretty_print()

================================ Human Message =================================

Explain machine learning.
================================== Ai Message ==================================

Machine learning is a subset of AI where computers learn patterns from data without explicit programming. They improve performance on tasks through experience.


In [4]:
for msg in beginner_response['messages']:
    msg.pretty_print()

================================ Human Message =================================

Explain machine learning.
================================== Ai Message ==================================

Here’s a simple, beginner-friendly explanation of **machine learning**, broken down into core ideas and how it works.

---

### What is Machine Learning? (The Simple Analogy)

Imagine teaching a child to recognize a cat. You don't give the child a strict rulebook like: "A cat has pointy ears, whiskers, and a tail." Instead, you show the child many, many pictures of cats (and non-cats). Over time, the child's brain figures out the patterns on its own – like the shape of the ears, the texture of fur, the way cats move. Eventually, the child can look at a new photo they've never seen and say, "That's a cat!"

**Machine learning (ML)** is the same concept, but for computers. Instead of being explicitly programmed with step-by-step rules for every task, the computer is given a large amount of data and a 